In [1]:
import os

In [2]:
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\wine-quality\\notebooks'

In [3]:
os.chdir("../")
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\wine-quality'

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from dataclasses import dataclass
from pathlib import Path
from src.wine_quality_prediction.constants import *
from src.wine_quality_prediction.utils.common import read_yaml, create_directories
from src.wine_quality_prediction import logger

In [5]:
@dataclass(frozen=True)
class DataTransformationConfig:
    root_directory: Path
    data_directory: Path

In [7]:
class ConfigurationManager:
    def __init__(self, config_file_path=CONFIG_FILE_PATH,
                 params_file_path=PARAMS_FILE_PATH,
                 schema_file_path=SCHEMA_FILE_PATH):
        self.config_file_path = read_yaml(path_to_yaml=config_file_path)
        self.params_file_path = read_yaml(path_to_yaml=params_file_path)
        self.schema_file_path = read_yaml(path_to_yaml=schema_file_path)

        create_directories([self.config_file_path.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config_file_path.data_transformation

        create_directories([config.root_directory])

        data_transformation_config = DataTransformationConfig(
            root_directory=config.root_directory,
            data_directory=config.data_directory,
        )

        return data_transformation_config

In [11]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def data_splitting(self):
        data = pd.read_csv(self.config.data_directory)

        train, test = train_test_split(data)

        train.to_csv(os.path.join(self.config.root_directory, "train.csv"), index=False, header=True)
        test.to_csv(os.path.join(self.config.root_directory, "test.csv"), index=False, header=True)

        logger.info("Splitted Data into Train and Test Set")
        logger.info(f"Train Shape: {train.shape} and Test Shape: {test.shape}")

In [12]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.data_splitting()
except Exception as e:
    raise e

[2026-03-05 01:39:44,139] INFO: common: YAML file 'config\config.yaml' read successfully.]
[2026-03-05 01:39:44,141] INFO: common: YAML file 'params.yaml' read successfully.]
[2026-03-05 01:39:44,142] INFO: common: YAML file 'schema.yaml' read successfully.]
[2026-03-05 01:39:44,143] INFO: common: Directory 'artifacts' created successfully or already exists.]
[2026-03-05 01:39:44,144] INFO: common: Directory 'artifacts/data_transformation' created successfully or already exists.]
[2026-03-05 01:39:44,156] INFO: 2243021212: Splitted Data into Train and Test Set]
[2026-03-05 01:39:44,157] INFO: 2243021212: Train Shape: (1199, 12) and Test Shape: (400, 12)]
